In [10]:
import pandas as pd
import sys
import numpy as np

n_prot=5
path_to_5_prot_data_folder="/home/jkipen/raid_storage/GenSimoneData/5Prot/"

# 5 protein case
Here I write of how to understand the probabilities in the files of the 5 protein case.

## Generating the data
The following cells just take the data from previous code of simulations 

In [8]:
import sys
sys.path.append('/home/jkipen/ProtInfGPU/code/numpy')
from WrapperEMCrossValv3 import WrapperEMCrossValv3
path_dataset="/home/jkipen/fast_data/ProtInfGPU/data/5_Prot/"

In [9]:
WEMV=WrapperEMCrossValv3(path_dataset,n_crossval=2)
WEMV.gen_crossval_dists();
PY_ests=WEMV.compute_cv()

In [32]:
index_crossval=0; #We generated 2 runs for crossvalidations, we just keep the data from the first one.

In [35]:
P_F_given_X=WEMV.PFo_given_X_db[WEMV.Idxs_cv[index_crossval],:]

In [37]:
Fs_true=WEMV.Idxs_cv[index_crossval]//1000

In [36]:
P_prot_true=WEMV.P_Ys_true[index_crossval]

In [38]:
P_prot_est=PY_ests[index_crossval]

In [39]:
n_flus = 123
P_F_given_prot = np.zeros((n_prot, n_flus), dtype=float)

for i, flu_ids in enumerate(WEMV.PIEM.list_flu_exp_iz_per_I):
    P_F_given_prot[i, flu_ids] = WEMV.PIEM.list_p_flu_exp_per_I[i]

In [40]:
Expected_N_F_per_prot=WEMV.PIEM.flu_exp_count_per_prot

In [43]:
np.savez(path_to_5_prot_data_folder+"DataOut.npz",P_F_given_X=P_F_given_X,Fs_true=Fs_true,P_prot_true=P_prot_true,P_prot_est=P_prot_est,P_F_given_prot=P_F_given_prot,Expected_N_F_per_prot=Expected_N_F_per_prot)

## Recovering data

### Data that was just copied, used for general settings

First I start from the database UP000005640_9606.fasta. Then, having selected the proteins the proteins that I mentioned in the paper (i think they were PSME4, HBA2, HBB, PSME3, and PSMB10). Then using this code from my github: "ProtInfGPU/code/exporting/whatprot/gen5Prot.py", which uses scripts from whatprot, it checks which peptides are generated and the sequences of fluorescence they produce. NOTE: Here is important that a lot of peptides produce no fluorescence reads and some peptides are ommitted! The output is basically dye-seqs.tsv and ExpTable.csv:

In [3]:
exp=pd.read_csv(path_to_5_prot_data_folder+'ExpTable.csv')
exp

,Peptide Id,Peptide String,Original Protein Id,Flustring Id,Flustring
0,0,MLKPALEPR,0,0,0......
1,1,GGFSFENCQR,0,1,1.0.....
2,2,NASLER,0,2,0....
3,6,TGTTIAGLVFQDGVILGADTR,0,3,0......0...........
4,7,ATNDSVVADK,0,4,0....0...
...,...,...,...,...,...
163,269,EMAATTLSGLLQCNFLTMDSPMQIHFEQLCK,4,118,1..0.......0.....1...........0
164,276,DPGSVGDTIPSAELVK,4,119,0.....0.....0
165,278,HAGVLGLGACVLSSPYDVPTWMPQLLMNLSAHLNDPQPIEMTVK,4,120,0....0.................02.....1.........
166,282,THHDNWQEHK,4,121,0...0...


Notice that for each column we get the peptide string, from which protein id came (id of protein to protein name is given in protein_descriptions_5prot.csv). Flustring is the fluorescence sequence it produces, and by looking at the python file that generated I had set up the labelling to "label_set = ['DE','C','Y']": So D or E become 0, C 1 and Y 2. All other aminoacids are ".", and the flustring is the peptide mirrored, then converted to number or "." and any "." before a number is removed since the system does not detect it.  

Notice that the peptide idx goes up because some peptides do not produce any fluorescence sequence. Also there can be many peptide to fluoroseq mapping (see peptide id 22 also maps to flustring 0). 

In [ ]:
exp.iloc[10:,:]

,Peptide Id,Peptide String,Original Protein Id,Flustring Id,Flustring
10,18,LPFTALGSGQDAALAVLEDR,0,10,00......0..........
11,19,FQPNMTLEAAQGLLVEAVTAGILGDLGSGGNVDACVITK,0,11,1.0.......0........0.......0.......
12,22,TLSSPTEPVK,0,0,0......
13,25,YHFVPGTTAVLTQTVKPLTLELVEETVQAMEVE,0,12,0.0.....00..0...................2
14,27,VDQEVK,1,13,0.0.
...,...,...,...,...,...
163,269,EMAATTLSGLLQCNFLTMDSPMQIHFEQLCK,4,118,1..0.......0.....1...........0
164,276,DPGSVGDTIPSAELVK,4,119,0.....0.....0
165,278,HAGVLGLGACVLSSPYDVPTWMPQLLMNLSAHLNDPQPIEMTVK,4,120,0....0.................02.....1.........
166,282,THHDNWQEHK,4,121,0...0...


### Loading simulation data

Here it loads the data that had to be exported from the code of simulations


In [44]:
npzfile = np.load(path_to_5_prot_data_folder+"DataOut.npz")

This line loads the different files of data I generated from it.

In [45]:
npzfile.files

['P_F_given_X',
 'Fs_true',
 'P_prot_true',
 'P_prot_est',
 'P_F_given_prot',
 'Expected_N_F_per_prot']

These are the data that I think will be enough to run your methods.

In [46]:
npzfile['P_F_given_X']

array([[3.7400010e-01, 9.6546188e-03, 2.2040365e-02, ..., 0.0000000e+00,
        3.8536869e-03, 0.0000000e+00],
       [3.7400010e-01, 9.6546188e-03, 2.2040365e-02, ..., 0.0000000e+00,
        3.8536869e-03, 0.0000000e+00],
       [3.7400010e-01, 9.6546188e-03, 2.2040365e-02, ..., 0.0000000e+00,
        3.8536869e-03, 0.0000000e+00],
       ...,
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 4.7989664e-07,
        0.0000000e+00, 9.9733061e-01],
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 4.7989664e-07,
        0.0000000e+00, 9.9733061e-01],
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 4.7989664e-07,
        0.0000000e+00, 9.9733061e-01]], dtype=float32)

This is the matrix I understood you need from the email. Each row is a read and each column shows the probability that "probeam" estimated. But the probabilities are over fluorescence sequences, which dont map one on one to peptides. The mapping is shown in the table I showed before! In this case is 123 different fluorescence sequences for the 5 proteins. In the whole proteome this scales to 150 000, that is why I think it is not feasible to send you whole proteome data in this format.

In [47]:
npzfile['Fs_true']

array([  0,   0,   0, ..., 122, 122, 122], dtype=int32)

This array shows which was the "true" fluorescence sequence that generated the ith read of the previous matrix. Notice that they are ordered (it was just because it provided some simplifications in simulations).

In [48]:
npzfile['P_prot_true']

array([0.19749294, 0.14758619, 0.07122915, 0.54229789, 0.04139384])

These are the true protein distributions for this case.

In [52]:
npzfile['P_prot_est']

array([0.19818016, 0.14624763, 0.07231432, 0.54213349, 0.0411244 ])

These are the estimated protein abundances after 30 iterations of the protein inference method i had

In [50]:
npzfile['P_F_given_prot']

array([[0.13646339, 0.07300791, 0.06823169, 0.07300791, 0.07300791,
        0.07300791, 0.07336729, 0.06823169, 0.06823169, 0.07336565,
        0.07334225, 0.07336729, 0.0733674 , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.  

This matrix has 5 rows (each indicating a protein) and each column the relative amount of each fluorosequence it generates. NOTE: proteins generate very different amount of fluorosequences!

In [51]:
npzfile['Expected_N_F_per_prot']

array([ 13.63002951,  20.47496261,   9.769371  ,   6.78470898,
       112.73542727])

This last vector shows the expected amount of fluorescence sequences per protein. Protein 5 generates in avg 112 fluorescence sequences while protein 4 only generates 6.78 fluorescence signals per protein!